In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/JFF_formatted_public_release_labels.csv", encoding="latin1")
df.shape

(18429, 153)

In [3]:
drop_columns = [
    'SAMPLE', 'MODE', 'RESPONDENT_DATE', 'Q5', 'Q6_TEXT', 
    'Q7_CODED', 'Q7_TEXT', 'Q12', 'Q12_TEXT', 'Q13', 'Q15', 'Q15_TEXT', 'Q16_TEXT', 
    'Q26_1', 'Q26_2', 'Q26_3', 'Q26_4', 'Q26_5', 'Q26_6', 'Q26_7', 
    'Q29', 'Q37A', 'Q37B', 'Q37C', 'Q37D', 'Q37E', 'Q37F', 'Q37G', 'Q37H', 'Q37I',
    'Q38A', 'Q38B', 'Q38C', 'Q39', 'Q42', 'Q43', 'Q46', 'Q46_TEXT', 'Q47', 'Q48', 'Q49', 'Q49_TEXT', 'Q50',
    'Q55_TEXT', 'Q64_1', 'Q64_2', 'Q64_3', 'Q64_4', 'Q64_5', 'Q64_6', 'Q66_1', 'Q66_2', 'Q66_3', 'Q66_4', 'Q66_5', 'Q66_TEXT',
    'Q67', 'Q68', 'W2_STATUS', 'WEIGHT'
]

# Drop unwanted columns present in df, and drop missing values for target Q4
cols_to_drop = [col for col in drop_columns if col in df.columns]
df_clean = df.drop(columns=cols_to_drop).dropna(subset=["Q4"]).copy()

In [4]:
# Check unique values and data types in Q6
print(f"Total unique values in Q6: {df_clean['Q6'].nunique()}")
print("\nSample occupation entries:")
display(df_clean['Q6'].value_counts().head(20))

# If check text responses for unique jobs
if 'Q6_TEXT' in df.columns:
    print("\nSample free-text responses (Q6_TEXT):")
    display(df['Q6_TEXT'].dropna().head(10))

Total unique values in Q6: 26

Sample occupation entries:


Q6
Educational instruction and library               2028
Management                                        1984
Business and financial operations                 1907
Office and administrative support                 1472
Computer and mathematical                         1263
Sales and related                                 1208
Healthcare practitioners                           953
Healthcare support                                 735
Transportation and material moving                 692
Arts, design, entertainment, sports, and media     652
Architecture and engineering                       645
Production                                         576
Community and social service                       551
Installation, maintenance, and repair              547
Legal                                              546
Food preparation and service                       434
Life, physical, and social science                 405
Personal care and service                          369
Constru


Sample free-text responses (Q6_TEXT):


0                    
1                    
2                    
3    HOME IMPROVEMENT
4                 -98
5                 -98
6                    
7                 -98
8                    
9                    
Name: Q6_TEXT, dtype: str

In [5]:
# Load the Excel file
excel_path = "../data/Work Values.xlsx" 
df_raw_wv = pd.read_excel(excel_path)

# Filter for the Extent scale (EX) and drop the high-point ranking flags (VH)
df_ex = df_raw_wv[df_raw_wv['Scale ID'] == 'EX'].copy()

# Pivot long-format O*NET rows into wide format (one row per occupation, one column per work value)
df_onet = df_ex.pivot_table(index=['O*NET-SOC Code', 'Title'], columns='Element Name', values='Data Value').reset_index()

# Clean up column multi-index names
df_onet.columns.name = None

print(f"Formatted O*NET Work Values Dataset Shape: {df_onet.shape}")
display(df_onet.head(3))

Formatted O*NET Work Values Dataset Shape: (874, 8)


,O*NET-SOC Code,Title,Achievement,Independence,Recognition,Relationships,Support,Working Conditions
0,11-1011.00,Chief Executives,6.33,7.00,7.00,5.00,5.33,6.33
1,11-1011.03,Chief Sustainability Officers,6.67,6.67,6.00,5.00,3.33,6.33
2,11-1021.00,General and Operations Managers,5.33,6.00,5.67,6.33,4.67,6.00


In [8]:
# Map O*NET 2-Digit SOC Codes
def map_soc_to_ajq(soc_code):
    prefix = str(soc_code).strip()[:2]
    reverse_map = {
        '11': 'Management',
        '13': 'Business and financial operations',
        '15': 'Computer and mathematical',
        '17': 'Architecture and engineering',
        '19': 'Life, physical, and social science',
        '21': 'Community and social service',
        '23': 'Legal',
        '25': 'Educational instruction and library',
        '27': 'Arts, design, entertainment, sports, and media',
        '29': 'Healthcare practitioners',
        '31': 'Healthcare support',
        '33': 'Protective service',
        '35': 'Food preparation and service',
        '37': 'Building and grounds cleaning and maintenance',
        '39': 'Personal care and service',
        '41': 'Sales and related',
        '43': 'Office and administrative support',
        '45': 'Farming, fishing, and forestry',
        '47': 'Construction and extraction',
        '49': 'Installation, maintenance, and repair',
        '51': 'Production',
        '53': 'Transportation and material moving',
        '55': 'Military'
    }
    return reverse_map.get(prefix, 'Other')

df_onet['AJQ_Category'] = df_onet['O*NET-SOC Code'].apply(map_soc_to_ajq)     

# Aggregate O*NET work values by AJQ Category
work_value_cols = ['Achievement', 'Independence', 'Recognition', 'Relationships', 'Support', 'Working Conditions']
df_onet_aggregated = df_onet.groupby('AJQ_Category')[work_value_cols].mean().reset_index()

# Duplicate the 'Healthcare practitioners' row for 'Healthcare technicians'
hc_row = df_onet_aggregated[df_onet_aggregated['AJQ_Category'] == 'Healthcare practitioners'].copy()
if not hc_row.empty:
    hc_row['AJQ_Category'] = 'Healthcare technicians'
    df_onet_aggregated = pd.concat([df_onet_aggregated, hc_row], ignore_index=True)

# Duplicate the 'Protective service' row as a proxy for 'Military'
mil_row = df_onet_aggregated[df_onet_aggregated['AJQ_Category'] == 'Protective service'].copy()
if not mil_row.empty:
    mil_row['AJQ_Category'] = 'Military'
    df_onet_aggregated = pd.concat([df_onet_aggregated, mil_row], ignore_index=True)

print("Aggregated O*NET Profiles by AJQ Category:")
display(df_onet_aggregated.head())

# Merge onto AJQ survey data
df_ajq_with_onet = pd.merge(
    df_clean,
    df_onet_aggregated,
    left_on='Q6',
    right_on='AJQ_Category',
    how='left'
)

# Confirm row counts and check coverage rate
print(f"\nOriginal df_clean rows: {len(df_clean)}")
print(f"Merged df_ajq_with_onet rows: {len(df_ajq_with_onet)}")

matched_count = df_ajq_with_onet['Achievement'].notnull().sum()
coverage_rate = (matched_count / len(df_ajq_with_onet)) * 100
print(f"O*NET Coverage Rate in AJQ: {coverage_rate:.2f}%")

Aggregated O*NET Profiles by AJQ Category:


,AJQ_Category,Achievement,Independence,Recognition,Relationships,Support,Working Conditions
0,Architecture and engineering,5.117818,5.020909,4.716909,3.667636,4.592000,5.101091
1,"Arts, design, entertainment, sports, and media",5.206316,5.026842,4.360263,4.495526,3.395263,4.360000
2,Building and grounds cleaning and maintenance,2.710000,3.791250,2.207500,4.248750,4.083750,3.000000
3,Business and financial operations,4.747907,4.918140,4.221628,4.690000,4.264186,4.530465
4,Community and social service,5.429286,5.142143,4.262143,6.500000,4.048571,4.630714



Original df_clean rows: 18429
Merged df_ajq_with_onet rows: 18429
O*NET Coverage Rate in AJQ: 99.55%


In [9]:
# Check random samples showing Q6 (AJQ Category) alongside the merged O*NET Work Values
validation_cols = ['Q6', 'AJQ_Category', 'Achievement', 'Independence', 'Working Conditions']
display(df_ajq_with_onet[validation_cols].dropna().sample(10, random_state=42))

,Q6,AJQ_Category,Achievement,Independence,Working Conditions
12616,Office and administrative support,Office and administrative support,2.986471,3.420784,3.212549
4329,Transportation and material moving,Transportation and material moving,3.022553,3.888723,3.456809
10002,Building and grounds cleaning and maintenance,Building and grounds cleaning and maintenance,2.710000,3.791250,3.000000
11970,Food preparation and service,Food preparation and service,2.500625,3.135000,2.463750
10545,Educational instruction and library,Educational instruction and library,5.463333,5.424561,5.054211
14503,Community and social service,Community and social service,5.429286,5.142143,4.630714
4435,Management,Management,5.385294,5.610392,5.378824
15832,Legal,Legal,5.190000,4.808571,4.998571
13894,Business and financial operations,Business and financial operations,4.747907,4.918140,4.530465
12768,Healthcare support,Healthcare support,3.649474,3.404211,3.491579


In [10]:
# Identify rows where the merge resulted in NaN for O*NET features
unmatched_jobs = df_ajq_with_onet[df_ajq_with_onet['Achievement'].isnull()]['Q6'].value_counts()
print("Unmatched or Null Q6 Categories:")
display(unmatched_jobs)

Unmatched or Null Q6 Categories:


Q6
Something else (please specify):    66
No response                         17
Name: count, dtype: int64

In [12]:
# Extract SOC_2Digit if not already present in df_onet
if 'SOC_2Digit' not in df_onet.columns:
    df_onet['SOC_2Digit'] = df_onet['O*NET-SOC Code'].astype(str).str.strip().str[:2]

# Summary comparison of the job fields
print("--- AJQ Unique Job Categories (Q6) ---")
print(df_clean['Q6'].unique())

print("\n--- Sample O*NET Granular Titles mapped to SOC Prefixes ---")
display(df_onet[['O*NET-SOC Code', 'Title', 'SOC_2Digit', 'AJQ_Category']].head(10))

--- AJQ Unique Job Categories (Q6) ---
<StringArray>
[ 'Building and grounds cleaning and maintenance',
              'Business and financial operations',
                      'Computer and mathematical',
                    'Construction and extraction',
              'Office and administrative support',
          'Installation, maintenance, and repair',
                         'Healthcare technicians',
                       'Healthcare practitioners',
            'Educational instruction and library',
                             'Protective service',
               'Something else (please specify):',
                                     'Management',
             'Life, physical, and social science',
                              'Sales and related',
                             'Healthcare support',
                                     'Production',
             'Transportation and material moving',
                   'Architecture and engineering',
                   'Food prep

,O*NET-SOC Code,Title,SOC_2Digit,AJQ_Category
0,11-1011.00,Chief Executives,11,Management
1,11-1011.03,Chief Sustainability Officers,11,Management
2,11-1021.00,General and Operations Managers,11,Management
3,11-1031.00,Legislators,11,Management
4,11-2011.00,Advertising and Promotions Managers,11,Management
5,11-2021.00,Marketing Managers,11,Management
6,11-2022.00,Sales Managers,11,Management
7,11-3012.00,Administrative Services Managers,11,Management
8,11-3021.00,Computer and Information Systems Managers,11,Management
9,11-3031.00,Financial Managers,11,Management


In [13]:
df_ajq_with_onet.head(5)

,ENTITY_ID,EndDate,S1,Q1,Q2,Q3,Q4,Q6,Q6_CODED,Q7,...,REGION,DIVISION,WORKER_TYPE,AJQ_Category,Achievement,Independence,Recognition,Relationships,Support,Working Conditions
0,4398199396,,,7,7,Very good,6,Building and grounds cleaning and maintenance,,Health/medical,...,Northeast,New England,Employee,Building and grounds cleaning and maintenance,2.710000,3.791250,2.207500,4.248750,4.083750,3.000000
1,4398199398,,,10-Completely satisfied,8,Excellent,10-Completely satisfied,Business and financial operations,,Education,...,Northeast,New England,Employee,Business and financial operations,4.747907,4.918140,4.221628,4.690000,4.264186,4.530465
2,4398199412,,,8,8,Excellent,8,Computer and mathematical,,Health/medical,...,Northeast,New England,Employee,Computer and mathematical,5.322759,4.930690,4.568276,3.540345,4.321034,5.066552
3,4398199417,,,8,8,Very good,9,Construction and extraction,Construction and extraction,Construction,...,Northeast,New England,Self-employed IC,Construction and extraction,2.773279,3.420984,2.433934,3.765082,4.661148,3.260656
4,4398199431,1/16/2025 6:47:11,Yes,6,7,Very good,5,Office and administrative support,,Government,...,Northeast,New England,Employee,Office and administrative support,2.986471,3.420784,2.708824,4.644510,4.394314,3.212549
